**1) Импорт библиотек и объявление основных переменных:**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

CSV_PATH = "./health_fitness_dataset.csv"
USER_COL = "participant_id"
DATE_COL = "date"
TARGET_COL = "fitness_level"

# количество дней для регрессии прогресса
HORIZON_DAYS = 30
# количество дней, по которым будем предсказывать
L = 14

# список колонок для onehot
CATEGORICAL_CANDIDATES = ["gender", "activity_type", "intensity", "smoking_status", "health_condition"]

**2) Чтение датасета и первый этап обработки:**

In [ ]:
df = pd.read_csv(CSV_PATH)

# преобразование дат
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[USER_COL, DATE_COL, TARGET_COL])

# соритровка: необходима для merge_asof
df = df.sort_values([USER_COL, DATE_COL]).reset_index(drop=True)

**3) Подготовка целевой переменной:**

In [ ]:
# создание колонки с "+>30 дней относительно текущей"
df["future_date"] = df[DATE_COL] + pd.to_timedelta(HORIZON_DAYS, unit="D")

# создание таблицы с "будущими" измерениями
future_tbl = df[[USER_COL, DATE_COL, TARGET_COL]].rename(
    columns={DATE_COL: "future_meas_date", TARGET_COL: "fitness_level_future"}
).sort_values([USER_COL, "future_meas_date"])

# добавление к каждой текущей дате одного человека будущей
df = pd.merge_asof(
    left=df.sort_values([USER_COL, "future_date"]),
    right=future_tbl,
    left_on="future_date",
    right_on="future_meas_date",
    by=USER_COL,
    direction="forward",
    allow_exact_matches=True,
)

# удаление строк, в которых не оказалось "будущих" данных
df = df.dropna(subset=["fitness_level_future"]).reset_index(drop=True)

# определение target
df["y_delta"] = df["fitness_level_future"] - df[TARGET_COL]

# определение, выросло ли значение или нет
df["y_grow"] = (df["y_delta"] > 0).astype("int64")

**4) Преобразование всех данных в числа и onehot:**

In [ ]:
# удаление target колонок, чтобы они не использовались как признаки
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
drop_num = {TARGET_COL, "fitness_level_future", "y_delta", "y_grow"}
num_cols = [c for c in num_cols if c not in drop_num]

# сопоставление указанных ранее и реально существующих категориальных колонок
cat_cols = [c for c in CATEGORICAL_CANDIDATES if c in df.columns]

# датафрейм с id людей, датой и выбранными признаками
feat_df = df[[USER_COL, DATE_COL] + num_cols + cat_cols].copy()

# onehot категорий
if cat_cols:
    feat_df = pd.get_dummies(feat_df, columns=cat_cols, dummy_na=True)

# список финальных признаковых колонок для модели
row_feature_cols = [c for c in feat_df.columns if c not in [USER_COL, DATE_COL]]

**5) Оконное представление данных (временной отрезок в один вектор):**

In [ ]:
def build_windows_for_user(user_block: pd.DataFrame):
    user_block = user_block.sort_values(DATE_COL)

    # матрица признаков в float32
    ub_feat = feat_df.loc[user_block.index, row_feature_cols].to_numpy(dtype=np.float32)

    # target в числовые типы в numpy массив, даты в numpy массив
    y_delta = user_block["y_delta"].to_numpy(dtype=np.float32)
    y_grow = user_block["y_grow"].to_numpy(dtype=np.int64)
    dates = user_block[DATE_COL].to_numpy()

    # если данных о пользователе меньше L, вернуть none
    if len(user_block) < L:
        return None

    X_list, yd_list, yg_list, d_list = [], [], [], []

    # перебор всех конечных позиций окна
    for i in range(L - 1, len(user_block)):
        window = ub_feat[i - L + 1 : i + 1]
        # добавление в список окон как длинных векторов
        X_list.append(window.reshape(-1))
        # сохранение целевых переменных для последней записи окна
        yd_list.append(y_delta[i])
        yg_list.append(y_grow[i])
        # сохранение даты целевой переменной
        d_list.append(dates[i])

    return (np.stack(X_list), np.array(yd_list), np.array(yg_list), np.array(d_list))

Xs, yds, ygs, gids, end_dates = [], [], [], [], []

for pid, g in df.groupby(USER_COL, sort=False):
    # построение окна для каждого пользователя
    out = build_windows_for_user(g)
    if out is None:
        continue
    # Xg: матрица окон формы (n_windows_user, L(количество дней)*F(количество признаков))
    # ydg: массив y_delta длины n_windows_user
    # ygg: массив y_grow длины n_windows_user
    # dg: массив дат конца окна длины n_windows_user
    Xg, ydg, ygg, dg = out
    Xs.append(Xg)
    yds.append(ydg)
    ygs.append(ygg)
    # присвоение каждому окну id человека
    gids.append(np.full(len(ydg), pid, dtype=object))
    end_dates.append(dg)

# объединение массивов в один датасет
X = np.concatenate(Xs, axis=0)
y_delta = np.concatenate(yds, axis=0)
y_grow = np.concatenate(ygs, axis=0)
groups = np.concatenate(gids, axis=0)
end_dates_arr = np.concatenate(end_dates, axis=0)

print("Prepared windowed data:")
print("X:", X.shape, " (n_samples, L*F)")
print("y_delta:", y_delta.shape, "  y_grow:", y_grow.shape)

**6) Стандартизация и сплит на train/test по людям:**

In [ ]:
# сплит на трейн, тест по группам
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
# индексы строк трейн, тест (строгое деление, чтобы данные одного пользователя не оказались и в трейн, и в тест)
train_idx, test_idx = next(gss.split(X, y_delta, groups=groups))

# деление окон и целевых признаков на трейн, тест
X_train, X_test = X[train_idx], X[test_idx]
y_delta_train, y_delta_test = y_delta[train_idx], y_delta[test_idx]
y_grow_train, y_grow_test = y_grow[train_idx], y_grow[test_idx]

# стандартизация и приведение к float32
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)